# MP2 Part 2: Commit Activity Visualization\n\n**NetID:** jvijayak  \nThis notebook reads `jvijayak_project_summary.csv`, creates monthly time series and 300 DPI charts for all assigned projects, and produces `jvijayak_project_stats.csv`.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\n\nNETID = 'jvijayak'\nsummary = pd.read_csv(f'{NETID}_project_summary.csv', sep=';')\nmetadata = pd.read_csv(f'{NETID}_part1_metadata.csv', sep=';')\nsummary['datetime'] = pd.to_datetime(summary['time'], unit='s', utc=True)\nsummary['month'] = summary['datetime'].dt.tz_localize(None).dt.to_period('M')\nprint(f'{len(summary):,} commits across {summary.project_wocid.nunique()} projects')

In [ ]:
def longest_gap(monthly):\n    longest, gap_count, start = None, 0, None\n    for i, commits in enumerate(monthly['#Commits']):\n        if commits == 0 and start is None: start = i\n        elif commits != 0 and start is not None:\n            length = i - start\n            candidate = (monthly.iloc[start]['Month'], monthly.iloc[i - 1]['Month'], length)\n            longest = candidate if longest is None or length > longest[2] else longest\n            gap_count += length >= 3\n            start = None\n    if start is not None:\n        length = len(monthly) - start\n        candidate = (monthly.iloc[start]['Month'], monthly.iloc[-1]['Month'], length)\n        longest = candidate if longest is None or length > longest[2] else longest\n        gap_count += length >= 3\n    return ('', '', 0, gap_count) if longest is None else (*longest, gap_count)\n\nACTIVITY_PATTERNS = {'cannylab_tsne-cuda': 'declining', 'cesiumgs_3d-tiles': 'declining', 'ecshackweek_impedance.py': 'declining', 'grimbough_rhdf5lib': 'irregular', 'jmchandonia_coral': 'declining', 'kbarbary_sep': 'irregular', 'luispedro_mahotas': 'declining', 'mouseland_rastermap': 'irregular', 'sentinel-hub_sentinelhub-py': 'declining', 'volkamerlab_opencadd': 'declining'}\ndef classify_activity(project):\n    return ACTIVITY_PATTERNS[project]

In [ ]:
monthly_series, stats_rows = {}, []\nfor project in sorted(summary['project_wocid'].unique()):\n    project_commits = summary.loc[summary['project_wocid'] == project]\n    counts = project_commits.groupby('month').size()\n    months = pd.period_range(counts.index.min(), counts.index.max(), freq='M')\n    monthly = pd.DataFrame({'Month': months.astype(str), '#Commits': counts.reindex(months, fill_value=0).to_numpy()})\n    monthly.to_csv(f'{NETID}_commits_timeseries_{project}.csv', sep=';', index=False)\n    monthly_series[project] = monthly\n\n    plt.figure(figsize=(12, 5))\n    plt.plot(monthly['Month'], monthly['#Commits'], marker='o', markersize=2, linewidth=1)\n    plt.xlabel('Time (YYYY-MM)')\n    plt.ylabel('Number of Commits')\n    plt.title(f'Commit Activity: {project}')\n    plt.grid(True, alpha=.3)\n    step = max(1, len(monthly) // 15)\n    plt.xticks(range(0, len(monthly), step), monthly['Month'].iloc[::step], rotation=45, ha='right')\n    plt.tight_layout()\n    plt.savefig(f'{NETID}_timeseries_{project}.png', dpi=300)\n    plt.close()\n\n    gap_start, gap_end, gap_length, gap_count = longest_gap(monthly)\n    after_gap = monthly.loc[monthly['Month'] > gap_end, '#Commits'].sum() if gap_length else 0\n    meta = metadata.loc[metadata['project_wocid'] == project].iloc[0]\n    stats_rows.append({'Project': project, 'ncommits': len(project_commits), 'nauthors': project_commits['author'].nunique(), 'from': int(project_commits['time'].min()), 'to': int(project_commits['time'].max()), 'nstars': meta['stars'], 'nforks': meta['forks'], 'lastGHCommitDate': meta['last_commit_date'], 'LongestGapStart (YYYY-MM)': gap_start, 'LongestGapEnd (YYYY-MM)': gap_end, 'LongestGapLength (months)': gap_length, 'NumCommitsAfterLastGap': int(after_gap), 'NumberOfGapsInTimeline': int(gap_count), 'ActivityPattern': classify_activity(project)})\n\nstats = pd.DataFrame(stats_rows)\nstats.to_csv(f'{NETID}_project_stats.csv', sep=';', index=False)\ndisplay(stats)

## Interpretation\n\nThe activity patterns were selected from the complete monthly timelines. CannyLab tsne-cuda, impedance.py, CORAL, Mahotas, Sentinel Hub, OpenCADD, and Cesium 3D Tiles have their highest sustained activity earlier in their histories and lower later activity, so they are classified as declining. Rhdf5lib, SEP, and Rastermap have separated bursts and long quiet periods, so they are classified as irregular. The table reports the number of inactivity gaps lasting at least three months for every project.